In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [3]:
# %pip install "sliderule>=5.6,<6" icesat2db geopandas pyproj numpy pandas scipy scikit-learn matplotlib pyyaml
# Restart the kernel if upgrading SlideRule.

from pathlib import Path
import copy
import pandas as pd
import yaml
from IPython.display import display
from src.remote_parallel_blocks import load_sites, run_sites
import icesat2db as isdb

### Settings

In [10]:
PROCESSING_CONFIG_PATH = Path("../config.yaml")
SITES_PATH = Path("../../scratch/climate_site_selection/regional_site_selection/"\
                    "candidates_ce67aa33f7ea25a1/selection_aec51d75bae2d8e0/n09/selected_sites.csv")
START_TIME = "2018-10-01T00:00:00Z"
END_TIME = "2019-01-01T00:00:00Z"
LENGTH_M = 30                      # Or 15.
SITE_INDEX = [0]                    # Position in the selected-site table.
MAX_RESOURCES = 5                 # Overpasses for this initial run.
GRAPH_DETAIL = 1                  # Eg/Ev scatter and fit; 0 disables plots.
RESOURCES = None                  # Discover overpasses in the date window.
# For an Ontario footprint, optionally use the known test acquisition:
# RESOURCES = ["ATL03_20181024210256_04010102_007_01.h5"]
# Explicit resources bypass date discovery and must intersect the chosen site.


### Load configuration and Inspect Sites

In [11]:
with PROCESSING_CONFIG_PATH.open() as stream:
    cfg = yaml.safe_load(stream)
remote_cfg = copy.deepcopy(cfg)
remote_cfg["parallel_blocks"]["rebinned"] = LENGTH_M
remote_cfg["parallel_blocks"]["DW"] = False  # Use ATL08 landcover for this remote run.

selected = load_sites(SITES_PATH)
display(selected[["site_id", "west", "south", "east", "north"]])


,site_id,west,south,east,north
0,S01,75.4,55.7,82.2,59.4
1,S02,48.4,62.9,57.0,66.6
2,S03,-151.1,62.0,-142.8,65.7
3,S04,116.5,68.3,127.2,72.0
4,S05,-62.1,48.5,-56.3,52.2
5,S06,115.6,53.0,121.9,56.7
6,S07,-104.7,53.0,-98.4,56.7
7,S08,115.5,59.3,123.1,63.0
8,S09,-77.0,54.8,-70.3,58.5


### Run the chosen site

In [13]:
runs = run_sites(
    selected.iloc[[SITE_INDEX]], remote_cfg, START_TIME, END_TIME,
    resources=RESOURCES, max_resources=MAX_RESOURCES,
    graph_detail=GRAPH_DETAIL,
)
run = next(runs, None)
if run is not None:
    display(run.diagnostics)
    display(run.results.head())
    display(run.segments.head())
else:
    print("No overpasses found in this window.")


ValueError: Buffer has wrong number of dimensions (expected 1, got 2)

# Inspect joined data

In [ ]:
if run is not None and not run.segments.empty:
    display(run.segments[[
        "beam", "bin_id", "photon_rate_te", "photon_rate_can_nr",
        "msw_flag", "layer_flag", "segment_landcover", "canopy_openness",
        "canopy_openness_resolution_m", "atl08_matched",
        "atl08_parent_count", "atl08_parent_fraction", "used_for_processing",
    ]].head(20))


# Optional: all sites and overpasses

In [ ]:
# result_tables = []
# for item in run_sites(selected, remote_cfg, START_TIME, END_TIME,
#                       max_resources=None, graph_detail=0):
#     display(item.diagnostics)
#     result_tables.append(item.results)
# results = pd.concat(result_tables, ignore_index=True) if result_tables else pd.DataFrame()
